# Spatial Block Bootstrap: RMSE Model Comparison


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

np.random.seed(35)


def find_project_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / 'scripts/forecasting_helpers.py').exists():
            return candidate
    raise FileNotFoundError('Could not locate project root from current working directory.')


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.forecasting_helpers import (
    TARGETS, CLIP_RANGES, PERSISTENCE_MAP,
    NFI12_TRAIN_INVNRS, NFI123_TRAIN_INVNRS, NFI12_TEST_INVNR, NFI123_TEST_INVNR,
    load_and_clean, build_model_inputs, load_best_hyperparams, train_scope,
    encode_features, predict_clip,
)

DATA_PATH = PROJECT_ROOT / 'data/preprocessed/final_nfi_ch2018_merged/NFI_with_Climate_Averages_RCP45.csv'
NFI12_HYPERPARAMS_PATH = PROJECT_ROOT / 'data/hyperparameters/nfi1/best_params.json'
NFI123_HYPERPARAMS_PATH = PROJECT_ROOT / 'data/hyperparameters/nfi12/best_params.json'
OUTPUT_DIR = PROJECT_ROOT / 'tables/rolling_window_eval'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SPATIAL_BLOCK_SIZES_M = [10_000, 20_000, 40_000]
N_SPATIAL_BOOT = 5000

## Rebuild direct and iterative held-out predictions


In [2]:
def direct_pointwise_predictions(df, scope, models, test_invnr, comparison):
    test_df = df[df['INVNR'] == test_invnr].copy()
    x_test = encode_features(test_df, scope['cats_map'], scope['final_features'])
    frames = []

    for target in TARGETS:
        low, high = CLIP_RANGES[target]
        frames.append(pd.DataFrame({
            'Comparison': comparison,
            'Target': target,
            'X': test_df['X'].to_numpy(),
            'Y': test_df['Y'].to_numpy(),
            'y_true': test_df[target].to_numpy(),
            'xgboost': predict_clip(models[target]['XGBoost'], x_test, target),
            'lasso': predict_clip(models[target]['Lasso'], x_test, target),
            'persistence': np.clip(test_df[PERSISTENCE_MAP[target]].to_numpy(), low, high),
        }))

    return pd.concat(frames, ignore_index=True)


def iterative_nfi5_pointwise_predictions(df, scope, models, comparison):
    df350_all = df[df['INVNR'] == 350]
    df450_all = df[df['INVNR'] == 450]
    common_clnr = set(df350_all['CLNR']).intersection(df450_all['CLNR'])
    df350_common = df350_all[df350_all['CLNR'].isin(common_clnr)].copy()
    df450_common = df450_all[df450_all['CLNR'].isin(common_clnr)].copy()

    x350 = encode_features(df350_common, scope['cats_map'], scope['final_features'])
    clnr350 = df350_common['CLNR'].values

    step1 = {}
    for model_name in ['XGBoost', 'Lasso']:
        step1[model_name] = {
            'BASFPH': predict_clip(models['BASFPH_next_INVNR'][model_name], x350, 'BASFPH_next_INVNR'),
            'HWSW_prop': predict_clip(models['HWSW_prop_next_INVNR'][model_name], x350, 'HWSW_prop_next_INVNR'),
        }
    step1['Persistence'] = {
        'BASFPH': np.clip(df350_common['BASFPH'].values, 0, 100),
        'HWSW_prop': np.clip(df350_common['HWSW_prop'].values, 0, 1),
    }

    def align_step1_to_450(values):
        return pd.Series(values, index=clnr350).reindex(df450_common['CLNR']).values

    frames = []
    for target in TARGETS:
        low, high = CLIP_RANGES[target]
        pred_cols = {}
        for model_name, col_name in [('XGBoost', 'xgboost'), ('Lasso', 'lasso'), ('Persistence', 'persistence')]:
            modified_450 = df450_common.copy()
            modified_450['BASFPH'] = align_step1_to_450(step1[model_name]['BASFPH'])
            modified_450['HWSW_prop'] = align_step1_to_450(step1[model_name]['HWSW_prop'])

            if model_name == 'Persistence':
                pred_cols[col_name] = np.clip(modified_450[PERSISTENCE_MAP[target]].values, low, high)
            else:
                x_modified = encode_features(modified_450, scope['cats_map'], scope['final_features'])
                pred_cols[col_name] = predict_clip(models[target][model_name], x_modified, target)

        frames.append(pd.DataFrame({
            'Comparison': comparison,
            'Target': target,
            'X': df450_common['X'].to_numpy(),
            'Y': df450_common['Y'].to_numpy(),
            'y_true': df450_common[target].to_numpy(),
            'xgboost': pred_cols['xgboost'],
            'lasso': pred_cols['lasso'],
            'persistence': pred_cols['persistence'],
        }))

    return pd.concat(frames, ignore_index=True)


df = load_and_clean(DATA_PATH)
nfi12_scope = build_model_inputs(df, NFI12_TRAIN_INVNRS)
nfi123_scope = build_model_inputs(df, NFI123_TRAIN_INVNRS)

# train_scope returns (models, importances); only the models are needed here.
nfi12_models, _ = train_scope(nfi12_scope, load_best_hyperparams(NFI12_HYPERPARAMS_PATH))
nfi123_models, _ = train_scope(nfi123_scope, load_best_hyperparams(NFI123_HYPERPARAMS_PATH))

pointwise_direct = pd.concat([
    direct_pointwise_predictions(df, nfi12_scope, nfi12_models, NFI12_TEST_INVNR, 'NFI12 direct NFI3->NFI4'),
    direct_pointwise_predictions(df, nfi123_scope, nfi123_models, NFI123_TEST_INVNR, 'NFI123 direct NFI4->NFI5'),
], ignore_index=True)
pointwise_iterative = iterative_nfi5_pointwise_predictions(
    df, nfi12_scope, nfi12_models, 'NFI12 iterative NFI3->NFI5'
)
pointwise_all = pd.concat([pointwise_direct, pointwise_iterative], ignore_index=True)

pointwise_all.groupby(['Comparison', 'Target']).size()

Comparison                  Target              
NFI12 direct NFI3->NFI4     BASFPH_next_INVNR       120
                            HWSW_prop_next_INVNR    120
NFI12 iterative NFI3->NFI5  BASFPH_next_INVNR       120
                            HWSW_prop_next_INVNR    120
NFI123 direct NFI4->NFI5    BASFPH_next_INVNR       120
                            HWSW_prop_next_INVNR    120
dtype: int64

## Spatial block bootstrap for pairwise RMSE differences


In [3]:
PAIRWISE_MODEL_COMPARISONS = [
    ('XGBoost', 'xgboost', 'Persistence', 'persistence'),
    ('Lasso', 'lasso', 'Persistence', 'persistence'),
    ('XGBoost', 'xgboost', 'Lasso', 'lasso'),
]


def rmse(values):
    return np.sqrt(np.mean(np.asarray(values) ** 2))


def rmse_delta(frame, candidate_col, benchmark_col):
    candidate_error = frame['y_true'] - frame[candidate_col]
    benchmark_error = frame['y_true'] - frame[benchmark_col]
    return rmse(benchmark_error) - rmse(candidate_error)


def spatial_block_ids(frame, block_size_m):
    x_block = np.floor(frame['X'].to_numpy() / block_size_m).astype(int)
    y_block = np.floor(frame['Y'].to_numpy() / block_size_m).astype(int)
    x_str = pd.Series(x_block, index=frame.index).astype(str)
    y_str = pd.Series(y_block, index=frame.index).astype(str)
    return x_str + '_' + y_str


def spatial_block_bootstrap_rmse(frame, candidate_col, benchmark_col, block_size_m, n_boot=5000, seed=35):
    rng = np.random.default_rng(seed)
    frame = frame.copy()
    frame['_spatial_block'] = spatial_block_ids(frame, block_size_m)
    grouped_indices = list(frame.groupby('_spatial_block').indices.values())
    n_blocks = len(grouped_indices)
    observed_delta = rmse_delta(frame, candidate_col, benchmark_col)

    boot_values = np.empty(n_boot)
    for i in range(n_boot):
        sampled = rng.integers(0, n_blocks, size=n_blocks)
        idx = np.concatenate([grouped_indices[j] for j in sampled])
        boot_values[i] = rmse_delta(frame.iloc[idx], candidate_col, benchmark_col)

    centered = boot_values - observed_delta
    if observed_delta >= 0:
        p_one_sided = np.mean(centered <= -observed_delta)
    else:
        p_one_sided = 1.0

    return {
        'Block_Size_m': block_size_m,
        'Spatial_Blocks': n_blocks,
        'Mean_Rows_per_Block': len(frame) / n_blocks,
        'Observed_Delta_RMSE': observed_delta,
        'Spatial_CI_low': np.quantile(boot_values, 0.025),
        'Spatial_CI_high': np.quantile(boot_values, 0.975),
        'Spatial_p_one_sided_Candidate_better': p_one_sided,
        'Bootstrap_Share_Delta_Positive': np.mean(boot_values > 0),
    }


def append_spatial_block_rows(rows, comparison, target, part):
    for candidate_name, candidate_col, benchmark_name, benchmark_col in PAIRWISE_MODEL_COMPARISONS:
        for block_size_m in SPATIAL_BLOCK_SIZES_M:
            row = {
                'Comparison': comparison,
                'Target': target,
                'Candidate_Model': candidate_name,
                'Benchmark_Model': benchmark_name,
                'Metric': 'Delta_RMSE',
                'N': len(part),
            }
            row.update(
                spatial_block_bootstrap_rmse(
                    part,
                    candidate_col,
                    benchmark_col,
                    block_size_m,
                    n_boot=N_SPATIAL_BOOT,
                    seed=35 + len(rows),
                )
            )
            rows.append(row)


spatial_rows = []
for (comparison, target), part in pointwise_all.groupby(['Comparison', 'Target']):
    append_spatial_block_rows(spatial_rows, comparison, target, part)

for target, part in pointwise_direct.groupby('Target'):
    append_spatial_block_rows(spatial_rows, 'Pooled direct NFI3->NFI5', target, part)

spatial_block_tests = pd.DataFrame(spatial_rows)
spatial_cols = [
    'Comparison', 'Target', 'Candidate_Model', 'Benchmark_Model', 'Metric', 'N',
    'Block_Size_m', 'Spatial_Blocks', 'Mean_Rows_per_Block', 'Observed_Delta_RMSE',
    'Spatial_CI_low', 'Spatial_CI_high', 'Spatial_p_one_sided_Candidate_better',
    'Bootstrap_Share_Delta_Positive',
]
spatial_block_tests = spatial_block_tests.reindex(columns=spatial_cols)

spatial_path = OUTPUT_DIR / 'spatial_block_bootstrap_rmse_model_comparison.csv'
spatial_block_tests.to_csv(spatial_path, index=False)
print('Saved:', spatial_path)
display(spatial_block_tests.round(6))


Saved: /Users/dearieder/Documents/BABF-forecasting-helper/tables/rolling_window_eval/spatial_block_bootstrap_rmse_model_comparison.csv


,Comparison,Target,Candidate_Model,Benchmark_Model,Metric,N,Block_Size_m,Spatial_Blocks,Mean_Rows_per_Block,Observed_Delta_RMSE,Spatial_CI_low,Spatial_CI_high,Spatial_p_one_sided_Candidate_better,Bootstrap_Share_Delta_Positive
0,NFI12 direct NFI3->NFI4,BASFPH_next_INVNR,XGBoost,Persistence,Delta_RMSE,120,10000,113,1.061947,-0.576334,-1.023128,-0.168590,1.0000,0.0020
1,NFI12 direct NFI3->NFI4,BASFPH_next_INVNR,XGBoost,Persistence,Delta_RMSE,120,20000,89,1.348315,-0.576334,-0.964740,-0.182424,1.0000,0.0016
2,NFI12 direct NFI3->NFI4,BASFPH_next_INVNR,XGBoost,Persistence,Delta_RMSE,120,40000,49,2.448980,-0.576334,-0.927922,-0.198425,1.0000,0.0020
3,NFI12 direct NFI3->NFI4,BASFPH_next_INVNR,Lasso,Persistence,Delta_RMSE,120,10000,113,1.061947,0.103180,-0.160826,0.342801,0.2124,0.7876
4,NFI12 direct NFI3->NFI4,BASFPH_next_INVNR,Lasso,Persistence,Delta_RMSE,120,20000,89,1.348315,0.103180,-0.128969,0.336597,0.1948,0.8052
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,Pooled direct NFI3->NFI5,HWSW_prop_next_INVNR,Lasso,Persistence,Delta_RMSE,240,20000,89,2.696629,0.001363,-0.000236,0.002938,0.0466,0.9534
68,Pooled direct NFI3->NFI5,HWSW_prop_next_INVNR,Lasso,Persistence,Delta_RMSE,240,40000,49,4.897959,0.001363,-0.000162,0.002818,0.0398,0.9602
69,Pooled direct NFI3->NFI5,HWSW_prop_next_INVNR,XGBoost,Lasso,Delta_RMSE,240,10000,113,2.123894,-0.017133,-0.022027,-0.012465,1.0000,0.0000
70,Pooled direct NFI3->NFI5,HWSW_prop_next_INVNR,XGBoost,Lasso,Delta_RMSE,240,20000,89,2.696629,-0.017133,-0.022288,-0.012377,1.0000,0.0000


## Block-size justification: spatial range of the RMSE loss differential

The block bootstrap is only valid if each block is at least as large as the spatial-correlation range of the quantity being averaged. For an RMSE comparison that quantity is the **per-plot squared-error differential** `d_i = (y_i - benchmark_i)^2 - (y_i - candidate_i)^2` (its block-mean is what the bootstrap resamples). We estimate the dependence range of `d_i` directly from the data with an empirical semivariogram and a distance-banded Moran's I correlogram, and check that it falls below the smallest block size (10 km). If the correlogram decays to ~0 well within 10 km, blocks of 10/20/40 km each contain independent units and the bootstrap CIs are trustworthy; the stable sensitivity sweep above then has a quantitative basis rather than just an empirical one.


In [4]:
def squared_error_differential(frame, candidate_col, benchmark_col):
    """Per-plot contribution to the RMSE delta (block-mean is what the bootstrap resamples)."""
    return ((frame['y_true'] - frame[benchmark_col]) ** 2
            - (frame['y_true'] - frame[candidate_col]) ** 2).to_numpy()


def spatial_correlogram(x, y, values, max_dist_m=50_000, bin_width_m=2_000, max_points=2500, seed=35):
    """Binned empirical semivariogram + distance-banded Moran's I for an irregular point set.

    O(n^2) in the number of points, so we subsample to `max_points` to bound cost.
    Returns the per-band table plus the subsample size used for the significance band.
    """
    rng = np.random.default_rng(seed)
    x, y, values = np.asarray(x, float), np.asarray(y, float), np.asarray(values, float)
    n = len(values)
    if n > max_points:
        sel = rng.choice(n, size=max_points, replace=False)
        x, y, values = x[sel], y[sel], values[sel]
    n_used = len(values)

    v = values - values.mean()
    var = np.mean(v ** 2)
    iu = np.triu_indices(n_used, k=1)
    dist = np.sqrt((x[:, None] - x[None, :]) ** 2 + (y[:, None] - y[None, :]) ** 2)[iu]
    prod = (v[:, None] * v[None, :])[iu]
    sq = ((values[:, None] - values[None, :]) ** 2)[iu]

    edges = np.arange(0, max_dist_m + bin_width_m, bin_width_m)
    rows = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (dist >= lo) & (dist < hi)
        if m.sum() < 50:
            continue
        rows.append({
            'dist_center_km': (lo + hi) / 2 / 1000,
            'n_pairs': int(m.sum()),
            'morans_I': prod[m].mean() / var,
            'semivariance': 0.5 * sq[m].mean(),
        })
    return pd.DataFrame(rows), n_used


def practical_range_km(correlogram, n_used):
    """Smallest band center where Moran's I drops below the ~2-sigma no-autocorrelation band."""
    cutoff = 2.0 / np.sqrt(n_used)
    if correlogram.empty or 'morans_I' not in correlogram.columns:
        return np.nan, cutoff
    below = correlogram.loc[correlogram['morans_I'] <= cutoff, 'dist_center_km']
    return (float(below.iloc[0]) if len(below) else np.nan), cutoff


range_rows = []
range_sources = list(pointwise_all.groupby(['Comparison', 'Target']))
range_sources.extend(
    ((('Pooled direct NFI3->NFI5', target), part) for target, part in pointwise_direct.groupby('Target'))
)

for (comparison, target), part in range_sources:
    for candidate_name, candidate_col, benchmark_name, benchmark_col in PAIRWISE_MODEL_COMPARISONS:
        d = squared_error_differential(part, candidate_col, benchmark_col)
        correlogram, n_used = spatial_correlogram(part['X'].to_numpy(), part['Y'].to_numpy(), d)
        prange, cutoff = practical_range_km(correlogram, n_used)
        range_rows.append({
            'Comparison': comparison,
            'Target': target,
            'Candidate_Model': candidate_name,
            'Benchmark_Model': benchmark_name,
            'Subsample_n': n_used,
            'Morans_I_band_0_2km': correlogram['morans_I'].iloc[0] if len(correlogram) else np.nan,
            'Significance_cutoff_I': cutoff,
            'Practical_Range_km': prange,
            'Smallest_Block_km': SPATIAL_BLOCK_SIZES_M[0] / 1000,
            'Block_exceeds_range': (np.isnan(prange)) or (prange <= SPATIAL_BLOCK_SIZES_M[0] / 1000),
        })

block_size_justification = pd.DataFrame(range_rows)
range_path = OUTPUT_DIR / 'spatial_block_size_justification.csv'
block_size_justification.to_csv(range_path, index=False)
print('Saved:', range_path)
print('Smallest block size:', SPATIAL_BLOCK_SIZES_M[0] / 1000, 'km')
display(block_size_justification.round(4))


Saved: /Users/dearieder/Documents/BABF-forecasting-helper/tables/rolling_window_eval/spatial_block_size_justification.csv
Smallest block size: 10.0 km


,Comparison,Target,Candidate_Model,Benchmark_Model,Subsample_n,Morans_I_band_0_2km,Significance_cutoff_I,Practical_Range_km,Smallest_Block_km,Block_exceeds_range
0,NFI12 direct NFI3->NFI4,BASFPH_next_INVNR,XGBoost,Persistence,120,NaN,0.1826,NaN,10.0,True
1,NFI12 direct NFI3->NFI4,BASFPH_next_INVNR,Lasso,Persistence,120,NaN,0.1826,NaN,10.0,True
2,NFI12 direct NFI3->NFI4,BASFPH_next_INVNR,XGBoost,Lasso,120,NaN,0.1826,NaN,10.0,True
3,NFI12 direct NFI3->NFI4,HWSW_prop_next_INVNR,XGBoost,Persistence,120,NaN,0.1826,NaN,10.0,True
4,NFI12 direct NFI3->NFI4,HWSW_prop_next_INVNR,Lasso,Persistence,120,NaN,0.1826,NaN,10.0,True
5,NFI12 direct NFI3->NFI4,HWSW_prop_next_INVNR,XGBoost,Lasso,120,NaN,0.1826,NaN,10.0,True
6,NFI12 iterative NFI3->NFI5,BASFPH_next_INVNR,XGBoost,Persistence,120,NaN,0.1826,NaN,10.0,True
7,NFI12 iterative NFI3->NFI5,BASFPH_next_INVNR,Lasso,Persistence,120,NaN,0.1826,NaN,10.0,True
8,NFI12 iterative NFI3->NFI5,BASFPH_next_INVNR,XGBoost,Lasso,120,NaN,0.1826,NaN,10.0,True
9,NFI12 iterative NFI3->NFI5,HWSW_prop_next_INVNR,XGBoost,Persistence,120,NaN,0.1826,NaN,10.0,True
